# Daily seafarer pipeline

This notebook creates one row per eligible ship and snapshot date. Run all cells from top to bottom, with this notebook opened in the project folder. It reads port snapshots and the sign-in CSV, then writes reports to `reports`.


## 1. Setup and helper functions

Requires `openpyxl`. The berth pattern is the same as in the existing bronze pipeline. Automatically collected snapshots include their full date and time. For older filenames such as `ships_in_port_OCT_24.xlsx`, set the year below.


In [ ]:
import argparse
import calendar
import csv
import itertools
import re
from collections import Counter, defaultdict
from datetime import datetime, date, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import openpyxl

BASE = Path.cwd()
BERTH = re.compile(r'APP[B-F]|HOLD|MKY1|SDW\d+|SDE\d+|SD\d+[WE]|BD24|WD[45]E|WDW[1-3]|WD[1-3]W|YVL5')
MONTHS = {name.upper(): n for n, name in enumerate(calendar.month_abbr) if name}

def snapshot_date(path, year):
    dated = re.search(r'portview_(\d{4}-\d{2}-\d{2})_', path.stem)
    if dated:
        return date.fromisoformat(dated.group(1))
    match = re.search(r'(?i)(?:^|[^a-z])(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[_ -]*(\d{1,2})(?:[^0-9]|$)', path.stem)
    if not match:
        raise ValueError(f'Cannot find month and day in snapshot filename: {path.name}')
    return date(year, MONTHS[match.group(1).upper()], int(match.group(2)))

def ship_key(value):
    # Parenthesized suffixes are retained: different names should not silently merge.
    return ' '.join(str(value or '').upper().split())

def write_csv(path, fields, rows):
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


## 2. Choose inputs

Update the year for the older files without a year in their names. The crew figure is an estimate per ship, as in your example.


In [ ]:
YEAR = 2026
CREW_PER_SHIP = 22
SNAPSHOTS = BASE / 'data' / 'port_snapshots'
SIGN_INS = BASE / 'data' / 'sign_ins' / 'MTSV Sign In Report.csv'
OUTPUT_DIR = BASE / 'reports'


## 3. Select ships at the specified berths

Read each snapshot, keep the berths in the existing filter, and combine duplicate entries for the same ship and date. A snapshot represents ships in the file on its filename date.


In [ ]:
files = sorted(p for p in SNAPSHOTS.iterdir() if p.suffix.lower() in ('.xlsx', '.xlsm') and not p.name.startswith('~$'))
if not files:
    raise ValueError('No port snapshots found')
ships = {}
dates = set()
for path in files:
    day = snapshot_date(path, YEAR)
    dates.add(day)
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        rows = wb.active.iter_rows(values_only=True)
        header = next(rows)
        labels = [str(x or '').strip().lower() for x in header]
        if 'berth' in labels and 'ship' in labels:
            bi, si = labels.index('berth'), labels.index('ship')
        elif len(header) == 11 and isinstance(header[7], (int, float)):
            bi, si = 2, 4
            rows = itertools.chain([header], rows)
        else:
            raise ValueError(f'{path.name}: unrecognised snapshot layout')
        for row in rows:
            berth = str(row[bi] or '').strip().upper()
            name = str(row[si] or '').strip()
            if BERTH.fullmatch(berth) and name:
                key = (day, ship_key(name))
                ships.setdefault(key, {'name': name, 'berths': set(), 'files': set()})
                ships[key]['berths'].add(berth)
                ships[key]['files'].add(path.name)
    finally:
        wb.close()


## 4. Count sign-ins by date and ship

Names are matched ignoring case and extra spaces; other differences are not automatically merged. Each sign-in row is counted once.

The latest automatic export for each day takes precedence over older exports and the original CSV, preventing repeated downloads from being counted twice.

Each name is counted once per ship per day, ignoring capital letters and extra spaces. Sign-ins with no name are excluded and flagged. Different spellings are treated as different people.


In [ ]:
counts = Counter()
unmatched = Counter()
sign_in_dates = set()
seen_people = set()
missing_names = 0
# Newest rolling export is authoritative for each covered day.
latest = {}
exports = SIGN_INS.parent / 'exports'
for path in sorted(exports.glob('infoodle_*.csv')) if exports.exists() else []:
    match = re.fullmatch(r'infoodle_(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})_(\d{8}T\d{6}Z)\.csv', path.name)
    if not match:
        continue
    start, end = date.fromisoformat(match[1]), date.fromisoformat(match[2])
    for day in dates:
        if start <= day <= end and (day not in latest or match[3] > latest[day][0]):
            latest[day] = (match[3], path)

# Historic CSV fills days not covered by an automatic export.
for path in [SIGN_INS, *sorted({item[1] for item in latest.values()})]:
    if not path.exists():
        continue
    with path.open(encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if not {'Time', 'Ship', 'Name'}.issubset(reader.fieldnames or []):
            raise ValueError(f'{path.name}: missing Time, Ship or Name column')
        for line, row in enumerate(reader, 2):
            if not any((value or '').strip() for value in row.values()):
                continue  # Infoodle exports blank separator rows.
            try:
                day = datetime.strptime(row['Time'].strip(), '%d/%m/%Y %H:%M:%S').date()
            except ValueError as e:
                raise ValueError(f'{path.name}, line {line}: invalid date {row["Time"]}') from e
            if day not in dates or (day in latest and latest[day][1] != path):
                continue
            sign_in_dates.add(day)
            name = (row['Ship'] or '').strip()
            key = (day, ship_key(name))
            person = ' '.join((row['Name'] or '').casefold().split())
            if not person:
                missing_names += 1
                continue
            person_key = (day, key[1], person)
            if person_key in seen_people:
                continue
            seen_people.add(person_key)
            if key in ships:
                counts[key] += 1
            else:
                unmatched[(day, name)] += 1

if missing_names:
    print(f'{missing_names} sign-in rows had no name and were excluded; check the source.')


## 5. Calculate and save the reports

Estimated seafarers = ship count × 22 by default. Percentage = sign-ins ÷ estimated seafarers. When the sign-in file has no records at all for a snapshot date, the counts are left blank because coverage is unknown. A date with some records may still be incomplete, so check the source before treating zeros as confirmed. Running this cell replaces the two output CSVs.

The report shows when each ship was last seen in a timestamped PortView snapshot, when the Infoodle export was collected, and when the report was refreshed (Melbourne time). Older files without timestamps are marked Unknown. A later Infoodle export can cover an earlier report date.


In [ ]:
# Times are local Melbourne time; historical files without a timestamp are marked unknown.
melbourne = ZoneInfo('Australia/Melbourne')
refreshed = datetime.now(melbourne).strftime('%Y-%m-%d %H:%M:%S')
result = []
for (day, key), info in sorted(ships.items()):
    count = counts[(day, key)]
    available = day in sign_in_dates
    observations = []
    for filename in info['files']:
        m = re.fullmatch(r'portview_(\d{4}-\d{2}-\d{2})_(\d{6})\.xlsx', filename)
        if m:
            observations.append(datetime.strptime(m[1] + m[2], '%Y-%m-%d%H%M%S'))
    port_time = max(observations).strftime('%Y-%m-%d %H:%M:%S') if observations else 'Unknown'
    export_time = datetime.strptime(latest[day][0], '%Y%m%dT%H%M%SZ').replace(tzinfo=timezone.utc).astimezone(melbourne).strftime('%Y-%m-%d %H:%M:%S') if day in latest else 'Unknown'
    result.append({'DATE': day.isoformat(), 'SEAFARERS SIGN IN': count if available else '', 'SHIP NAME': info['name'], 'SHIP COUNT': 1, 'TOTAL SEAFARERS': CREW_PER_SHIP, 'SEAFARERS %': f'{100 * count / CREW_PER_SHIP:.2f}%' if available else '', 'BERTH': ', '.join(sorted(info['berths'])), 'SOURCE SNAPSHOT': ', '.join(sorted(info['files'])), 'SIGN-IN STATUS': 'Records found for date' if available else 'No records for date; check source coverage', 'SHIP LAST SEEN (MELBOURNE)': port_time, 'INFOODLE EXPORT (MELBOURNE)': export_time, 'REPORT REFRESHED (MELBOURNE)': refreshed})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_csv(OUTPUT_DIR / 'daily_ship_report.csv', ['DATE', 'SEAFARERS SIGN IN', 'SHIP NAME', 'SHIP COUNT', 'TOTAL SEAFARERS', 'SEAFARERS %', 'BERTH', 'SOURCE SNAPSHOT', 'SIGN-IN STATUS', 'SHIP LAST SEEN (MELBOURNE)', 'INFOODLE EXPORT (MELBOURNE)', 'REPORT REFRESHED (MELBOURNE)'], result)
print(f'{len(result)} ship-day rows, {sum(counts.values())} matched sign-ins, {sum(unmatched.values())} unmatched sign-ins.')
print(f'Reports saved in {OUTPUT_DIR}')
